In [1]:
import matplotlib.pyplot as plt
import math
import matplotlib.patches as patches
import numpy as np
import matplotlib
import matplotlib.animation as anm    
from IPython.display import HTML   

In [2]:
class World:
    def __init__(self,time_span,time_interval,debug=False):    
        self.objects = []   
        self.debug = debug
        self.time_span = time_span    
        self.time_interval = time_interval    

    def append(self,obj):   
        self.objects.append(obj)

    def draw(self):
        fig = plt.figure(figsize=(4,4))    
        ax = fig.add_subplot(111)         
        ax.set_aspect('equal')            
        ax.set_xlim(-5,5)                 
        ax.set_ylim(-5,5)                  
        ax.set_xlabel("X",fontsize=10)     
        ax.set_ylabel("Y",fontsize=10)    

        elems = []  
        
        if self.debug:
            for i in range(1000):self.one_step(i,elems,ax)    
        else:
            self.ani = anm.FuncAnimation(fig,self.one_step,fargs=(elems,ax),
                                         frames=int(self.time_span/self.time_interval)+1,interval=int(self.time_interval*1000),repeat=False)
            plt.close()    
            return HTML(self.ani.to_jshtml()) 

    def one_step(self,i,elems,ax):    
        while elems:elems.pop().remove()  
        time_str = "t = %.2f[s]"%(self.time_interval*i)  
        elems.append(ax.text(-4.4, 4.5,"t = "+str(i),fontsize=10))    
        for obj in self.objects:
            obj.draw(ax,elems)
            if hasattr(obj,"one_step"):obj.one_step(self.time_interval)    

In [3]:
class IdealRobot:
    def __init__(self,pose,agent=None,sensor=None,color="black"):    #センサの指定を追加
        self.pose = pose    
        self.r = 0.2        
        self.color = color
        self.agent = agent    
        self.poses = [pose]   
        self.sensor = sensor

    def draw(self,ax,elems):   
        x,y,theta = self.pose    
        xn = x + self.r * math.cos(theta)    
        yn = y + self.r * math.sin(theta)    
        elems += ax.plot([x,xn],[y,yn],color=self.color)    
        c = patches.Circle(xy=(x,y),radius=self.r,fill=False,color=self.color)    
        elems.append(ax.add_patch(c))   

        self.poses.append(self.pose)    
        elems += ax.plot([e[0] for e in self.poses],[e[1] for e in self.poses],linewidth=0.5,color="black")

        if  self.sensor and len(self.poses) > 1:
            self.sensor.draw(ax,elems,self.poses[-2])    #センサ値を得た時刻の姿勢を渡す　-2は後ろから2番目（移動する前の姿勢）

    @classmethod   
    def state_transition(cls,nu,omega,time,pose):    
        t0 = pose[2]
        if math.fabs(omega) < 1e-10:    
            return pose + np.array([nu*math.cos(t0),
                                    nu*math.sin(t0),
                                    omega])*time
        else:
            return pose + np.array([nu/omega*(math.sin(t0+omega*time)-math.sin(t0)),
                                    nu/omega*(-math.cos(t0+omega*time)+math.cos(t0)),
                                    omega*time])

    def one_step(self,time_interval):   
        if not self.agent:return   
        obs = self.sensor.data(self.pose)
        nu, omega = self.agent.decision(obs)
        self.pose = self.state_transition(nu,omega,time_interval,self.pose)

In [4]:
class Agent:
    def __init__(self,nu,omega):
        self.nu = nu
        self.omega = omega

    def decision(self,observation=None):    #observationはセンサ値の受け渡しに使われる
        return self.nu,self.omega

In [5]:
class Landmark:
    def __init__(self,x,y):
        self.pos = np.array([x,y]).T
        self.id = None

    def draw(self,ax,elems):
        c = ax.scatter(self.pos[0],self.pos[1],s=100,marker="*",label="landmarks",color="orange")
        #scatterは散布図に点を打つためのメソッド　ランドマークの座標、点のサイズ（幅の二乗で指定）、点の形、ラベル、色を引数にしてる
        elems.append(c)
        elems.append(ax.text(self.pos[0],self.pos[1],"id:"+ str(self.id),fontsize=10))

In [6]:
class Map:
    def __init__(self):
        self.landmarks = []    #空のランドマークのリストを準備

    def append_landmark(self,landmark):    #ランドマークを追加
        landmark.id = len(self.landmarks)    #追加するランドマークにIDを与える
        self.landmarks.append(landmark)

    def draw(self,ax,elems):
        for lm in self.landmarks:lm.draw(ax,elems)    #landmarkのdrawを順に呼び出し、描画

In [7]:
class IdealCamera:
    def __init__(self,env_map,\
                distance_range=(0.5,6.0),
                direction_range=(-math.pi/3,math.pi/3)):    #ここの\は文が続いていることを表す
        self.map = env_map
        self.lastdata = []

        self.distance_range = distance_range
        self.direction_range = direction_range

    def visible(self,polarpos):
        if polarpos is None:
            return false

        return self.distance_range[0] <= polarpos[0] <= self.distance_range[1]\
            and self.direction_range[0] <= polarpos[1] <= self.direction_range[1]\

    def data(self,cam_pose):
        observed = []
        for lm in self.map.landmarks:
            z = self.observation_function(cam_pose,lm.pos)
            if self.visible(z):
                observed.append((z,lm.id))    #.append((a,b))はタプル型のオブジェクトを作るときの書き方で、aとbをペアにする

        self.lastdata = observed
        return observed

    @classmethod
    def observation_function(cls,cam_pose,obj_pos):    #cam_poseはロボットの姿勢 obj_posは物体の位置
        diff = obj_pos - cam_pose[0:2]    #cam_pose[0:2]はcam_poseの[0]~[1]のこと diff=[mjx-x mjy-y]
        phi = math.atan2(diff[1],diff[0])-cam_pose[2]    #cam_pose[2]はロボットの姿勢 Θ cam_pose=[x y Θ]
        while phi >= np.pi: phi -= 2*np.pi    # ←と↙はphiが[-π,π)の範囲に入るように正規化
        while phi < -np.pi: phi += 2*np.pi    # phi>=πの時 phi-2π、phi<-πの時 phi+2π
        return np.array([np.hypot(*diff),phi]).T    #np.hypot(x,y)は√(x^2 + y^2)を返す　*diffでdiffの各要素をhypotに渡す

    def draw(self,ax,elems,cam_pose):
        for lm in self.lastdata:    #世界座標系でのランドマークの位置を計算しなおし
            x,y,theta = cam_pose
            distanse,direction = lm[0][0],lm[0][1]
            lx = x + distanse * math.cos(direction + theta)
            ly = y + distanse * math.sin(direction + theta)
            elems += ax.plot([x,lx],[y,ly],color="pink")

In [8]:
world = World(10,0.1)    #コードにバグがあるときworld=World(debug=True)としてデバッグを行う

m = Map()    #　←地図を生成して　↓ランドマークを追加
m.append_landmark(Landmark(2,-2))
m.append_landmark(Landmark(-1,-3))
m.append_landmark(Landmark(3,3))
world.append(m)    #worldに地図を登録

straight = Agent(0.2,0.0)    
circling = Agent(0.2,10.0/180*math.pi)   

robot1 = IdealRobot(np.array([2,3,math.pi/6]).T,sensor=IdealCamera(m),agent=straight)    #エージェントstraight
robot2 = IdealRobot(np.array([-2,-1,math.pi/5*6]).T,sensor=IdealCamera(m),agent=circling,color="red")    #エージェントcircling

world.append(robot1)    
world.append(robot2)

world.draw()

In [9]:
cam = IdealCamera(m)
p = cam.data(robot2.pose)
print(p)

[(array([4.16661716, 0.92102083]), 0), (array([1.11539959, 0.53325976]), 1)]
